# 02 - nnU-Net v2 baselines (3d_fullres and Residual Encoder M)

Runtime -> **T4 GPU**. Uses the *same* train/val split as the custom models (fold 0 of `splits_final.json`);
test cases are held out in `imagesTs`. Raw/preprocessed data live on local disk (rebuilt each session, a few minutes);
**results/checkpoints live on Drive**. `nnUNetTrainer_UMD` caps epochs and saves `checkpoint_latest` every 5 epochs
(stock nnU-Net: every 50), and re-running the train cell resumes with `--c`.

ResEnc **M** is used because it fits a 16 GB T4; ResEnc L/XL need 24 GB+.

In [ ]:
VARIANT = '3d_fullres'   # '3d_fullres' (nnU-Net default plans) | 'resenc' (nnUNetResEncUNetMPlans)
SMOKE = True             # True: 3-epoch timing run into a separate smoke/ folder. False: the real run
EPOCHS = 250             # set from the smoke-test timing; identical for both variants and across sessions

In [ ]:
# --- shared setup: Drive + code ---------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/figo'          # shared project folder (one Colab account)
REPO_URL = 'https://github.com/acm-research-f26/Radii.git'  # public; if made private see README (token in Colab Secrets)
REPO = '/content/figo-repo'

import os, subprocess
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
%cd {REPO}
!pip -q install -r requirements.txt
!pip -q install nnunetv2

In [ ]:
import os
os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results'] = f'{DRIVE}/smoke/nnUNet_results' if SMOKE else f'{DRIVE}/nnUNet_results'
os.environ['UMD_NNUNET_EPOCHS'] = '3' if SMOKE else str(EPOCHS)
PLANS = 'nnUNetPlans' if VARIANT == '3d_fullres' else 'nnUNetResEncUNetMPlans'
PLANNER = [] if VARIANT == '3d_fullres' else ['-pl', 'nnUNetPlannerResEncM']

!python -m figo.nnunet convert --data-dir {DRIVE}/data/clean --splits {DRIVE}/data/splits.json \
    --nnunet-raw $nnUNet_raw --nnunet-preprocessed $nnUNet_preprocessed
!python -m figo.nnunet install-trainer

In [ ]:
# Preprocessing is CPU work: run up to this cell once on a CPU runtime; it is cached on Drive
# and later GPU sessions just unpack it (saves GPU units every session).
import subprocess
pre = f"/content/nnUNet_preprocessed/Dataset501_UMD/{PLANS}_3d_fullres"
tar = f"{DRIVE}/nnunet_cache/{PLANS}.tar"
if not os.path.isdir(pre):
    if os.path.exists(tar):
        !tar -xf {tar} -C /content/nnUNet_preprocessed
    else:
        subprocess.run(['nnUNetv2_plan_and_preprocess', '-d', '501', '-c', '3d_fullres',
                        '--verify_dataset_integrity', *PLANNER], check=True)
        os.makedirs(f'{DRIVE}/nnunet_cache', exist_ok=True)
        !tar -cf {tar} -C /content/nnUNet_preprocessed Dataset501_UMD
!cat /content/nnUNet_preprocessed/Dataset501_UMD/{PLANS}.json | python -c "import json,sys; c=json.load(sys.stdin)['configurations']['3d_fullres']; print('patch', c['patch_size'], 'spacing', c['spacing'], 'batch', c['batch_size'])"

In [ ]:
fold_dir = f"{os.environ['nnUNet_results']}/Dataset501_UMD/nnUNetTrainer_UMD__{PLANS}__3d_fullres/fold_0"
resume = ['--c'] if os.path.exists(f'{fold_dir}/checkpoint_latest.pth') else []
print('resuming' if resume else 'starting fresh', fold_dir)
!nnUNetv2_train 501 3d_fullres 0 -tr nnUNetTrainer_UMD -p {PLANS} {' '.join(resume)}

In [ ]:
# timing: nnU-Net logs one "Epoch time" line per epoch (the first is slower: warm-up)
!grep -h "Epoch time" {fold_dir}/training_log_*.txt
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv
!nproc

## Test-set inference and evaluation (after training)

In [ ]:
NAME = f'nnunet_{VARIANT}'
PRED = f'{DRIVE}/preds/{NAME}'
RES = f'{DRIVE}/results'
!nnUNetv2_predict -i $nnUNet_raw/Dataset501_UMD/imagesTs -o {PRED} -d 501 -c 3d_fullres -f 0 \
    -tr nnUNetTrainer_UMD -p {PLANS} -chk checkpoint_best.pth
!python -m figo.evaluate --pred-dir {PRED} --data-dir {DRIVE}/data/clean --splits {DRIVE}/data/splits.json \
    --subset test --out {RES}/{NAME}_seg.csv
!python -m figo.figo_eval agreement --ref-dir {DRIVE}/data/clean --pred-dir {PRED} \
    --splits {DRIVE}/data/splits.json --subset test --out {RES}/{NAME}_figo_agreement.csv